# 04. Prediction and Intervals

For a new predictor vector $\mathbf x_0$, the fitted mean response is

$$\widehat{\mu}_{Y|\mathbf x_0}=\mathbf x_0'\widehat{\boldsymbol\beta}.$$

A confidence interval estimates the mean response at that predictor setting. A prediction interval estimates a future individual response at that setting. The prediction interval is wider because it includes both uncertainty in the fitted mean and the noise of a new observation.


Let

$$\mathbf x_0=[1,x_{01},x_{02},\ldots,x_{0k}]'$$

and define the design-distance value

$$h_0=\mathbf x_0'(\mathbf X'\mathbf X)^{-1}\mathbf x_0.$$

Then the usual 100$(1-\alpha)$% intervals are

$$\text{CI for mean response: }\quad \widehat y_0 \pm t_{1-\alpha/2,\,n-k-1}\sqrt{MSE\,h_0},$$

$$\text{PI for a future response: }\quad \widehat y_0 \pm t_{1-\alpha/2,\,n-k-1}\sqrt{MSE(1+h_0)}.$$

The extra `1` in the prediction interval is the future-observation error variance.


In [ ]:
from lite_setup import ensure_packages
await ensure_packages()

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


In [ ]:
delivery = pd.read_csv("data/delivery_time.csv")
model = smf.ols("Time ~ Cases + Distance", data=delivery).fit()

new_job = pd.DataFrame({"Cases": [10], "Distance": [500]})
pred = model.get_prediction(new_job).summary_frame(alpha=0.05)
pred


In [ ]:
mean_hat = pred.loc[0, "mean"]
ci_low, ci_high = pred.loc[0, ["mean_ci_lower", "mean_ci_upper"]]
pi_low, pi_high = pred.loc[0, ["obs_ci_lower", "obs_ci_upper"]]

print(f"Expected delivery time: {mean_hat:.2f}")
print(f"95% CI for mean delivery time: [{ci_low:.2f}, {ci_high:.2f}]")
print(f"95% PI for one future delivery time: [{pi_low:.2f}, {pi_high:.2f}]")


In [ ]:
from checks import check_interval_order
print(check_interval_order(ci_low, mean_hat, ci_high, label="mean-response confidence interval"))
print(check_interval_order(pi_low, mean_hat, pi_high, label="individual prediction interval"))


## Why the Intervals Depend on Location

The term $\mathbf x_0'(\mathbf X'\mathbf X)^{-1}\mathbf x_0$ measures how far the new predictor setting is from the center of the observed design. Predictions near the center of the data are usually more precise than extrapolations.


In [ ]:
X = model.model.exog
x0 = np.array([1, new_job.loc[0, "Cases"], new_job.loc[0, "Distance"]])
h0 = x0 @ np.linalg.inv(X.T @ X) @ x0
print(f"Design distance h0 = {h0:.4f}")


Try changing `Cases` and `Distance` in `new_job`. Watch how the fitted mean and intervals respond, especially when you move far outside the observed data range.
